# Wind — Live Replay Pipeline: Export and Verify

The dashboard's **Live Replay** tab runs the trained XGBoost model on demand: raw readings in → features → forecast out.

This notebook:
1. Exports the raw readings the replay needs (late Oct–Dec 2018, cleaned exactly as in notebook 03)
2. **Verifies** that the dashboard's feature pipeline (`dashboard/wind_pipeline.py`) produces exactly the same features, and the same forecasts, as notebooks 03 and 04

Step 2 matters: a deployed model is only trustworthy if it receives exactly the features it was trained on. A silent mismatch would give wrong forecasts with no error message.

In [1]:
import sys
import json
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

# Use the same feature pipeline file as the dashboard
sys.path.append('../dashboard')
from wind_pipeline import build_features

# Clean the raw readings exactly as in notebook 03 (steps before feature creation)
raw = pd.read_csv('../data/raw/wind/T1.csv')
raw['Date/Time'] = pd.to_datetime(raw['Date/Time'], format='%d %m %Y %H:%M')
raw = raw.rename(columns={
    'Date/Time': 'timestamp',
    'LV ActivePower (kW)': 'power',
    'Wind Speed (m/s)': 'wind_speed',
    'Wind Direction (°)': 'wind_direction'
})
readings = (raw.drop_duplicates(subset='timestamp')
               .sort_values('timestamp')
               .set_index('timestamp')[['power', 'wind_speed', 'wind_direction']])

full_range = pd.date_range(readings.index.min(), readings.index.max(), freq='10min')
readings = readings.reindex(full_range)
readings.index.name = 'timestamp'
readings = readings.interpolate(method='time', limit=6)   # fill gaps up to 1 hour
readings['power'] = readings['power'].clip(lower=0)

print('Readings:', readings.shape)

Readings: (52560, 3)


## Export Replay Data

Readings from 30 October onwards: the test period (Nov–Dec) plus enough history before it for the 24-hour lag feature. The clipping limit used in training (maximum training power) is saved alongside the model.

In [2]:
replay = readings[readings.index >= '2018-10-30']
replay.to_csv('../data/processed/wind_replay_readings.csv')

wind_features = pd.read_csv('../data/processed/wind_features.csv', index_col='timestamp', parse_dates=True)
max_power = float(wind_features.loc[wind_features.index < '2018-09-01', 'power'].max())
with open('../models/wind_xgboost_meta.json', 'w') as f:
    json.dump({'max_power': max_power}, f)

print('Replay readings saved:', replay.shape)
print(f'Max training power (clipping limit): {max_power:.1f} kW')

Replay readings saved: (9072, 3)
Max training power (clipping limit): 3618.7 kW


## Check 1: Identical Features

The pipeline's features are compared with the features from notebooks 03–04 for every test-period row.

In [3]:
with open('../models/wind_xgboost_features.json') as f:
    FEATURES = json.load(f)

# Notebook 03 features for the test period, plus notebook 04's trend features
test_rows = wind_features[wind_features.index >= '2018-11-01'].copy()
test_rows['power_diff_1'] = test_rows['power'] - test_rows['power_lag_1']
test_rows['power_diff_6'] = test_rows['power'] - test_rows['power_lag_6']
test_rows['wind_speed_diff_1'] = test_rows['wind_speed'] - test_rows['wind_speed_lag_1']
test_rows['wind_speed_diff_6'] = test_rows['wind_speed'] - test_rows['wind_speed_lag_6']

pipeline = build_features(replay).loc[test_rows.index, FEATURES]

print('Test rows compared:', len(test_rows))
print('Rows where the pipeline produced missing values:', int(pipeline.isna().any(axis=1).sum()))
print('Largest feature difference:', (pipeline - test_rows[FEATURES]).abs().to_numpy().max())

Test rows compared: 8085
Rows where the pipeline produced missing values: 0
Largest feature difference: 2.5208355181050024e-08


## Check 2: Identical Forecasts

The saved model is run on the pipeline's features, and the forecasts are compared with the XGBoost test predictions saved in notebook 04.

In [4]:
model = XGBRegressor()
model.load_model('../models/wind_xgboost_delta.json')

replay_forecast = np.clip(pipeline['power'] + model.predict(pipeline), 0, max_power)

saved = pd.read_csv('../data/processed/wind_test_predictions.csv', index_col='timestamp', parse_dates=True)
difference = (replay_forecast - saved.loc[pipeline.index, 'xgboost_delta']).abs()

print(f'Largest forecast difference: {difference.max():.6f} kW')

Largest forecast difference: 0.000000 kW


## Result

- **Features:** all 8,085 test rows reproduced, with no missing values. Largest difference: 0.000000025 (rounding from saving to CSV).
- **Forecasts:** largest difference from notebook 04's saved predictions: **0.000000 kW**.

The dashboard's Live Replay runs exactly the same model on exactly the same features as the notebooks.